## Use collabrative filtering to train a track recommendation engine


In [31]:
import os

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

In [32]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType
from pyspark.sql.functions import when, col
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Create a Spark session and configure it to use the JDBC jar
spark = SparkSession.builder \
    .appName("spark collab") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .config("spark.jars", rel_path) \
    .config("spark.executor.extraClassPath", rel_path) \
    .config("spark.driver.extraClassPath", rel_path) \
    .getOrCreate()

In [ ]:
# Define the schema according to the provided JSON structure
schema = StructType([
    StructField("ts", LongType()),
    StructField("auth", StringType()),
    StructField("page", StringType()),
    StructField("song", StringType()),
    StructField("level", StringType()),
    StructField("artist", StringType()),
    StructField("gender", StringType()),
    StructField("method", StringType()),
    StructField("status", IntegerType()),
    StructField("userId", StringType()),
    StructField("lastName", StringType()),
    StructField("location", StringType()),
    StructField("track_id", IntegerType()),
    StructField("firstName", StringType()),
    StructField("sessionId", IntegerType()),
    StructField("userAgent", StringType()),
    StructField("registration", LongType()),
    StructField("itemInSession", IntegerType())
])

# Load the JSON data
data_path = "./data/music_log.json"  
df = spark.read.json(data_path, schema=schema)

# Display the DataFrame to verify it's loaded correctly
df.show(5)

In [ ]:
# Process the DataFrame to create a ratings column based on 'page' interactions
df = df.withColumn("rating", when(col("page") == "NextSong", 1).otherwise(0))

# Select only the necessary columns for ALS
ratings_df = df.select(col("userId").cast("integer"),
                       col("track_id").cast("integer"),
                       col("rating").cast("float"))

# Check the processed DataFrame
ratings_df.show()

In [ ]:
# Set up the ALS model
als = ALS(maxIter=5,
          regParam=0.01,
          userCol="userId",
          itemCol="track_id",
          ratingCol="rating",
          implicitPrefs=True)

# Fit the ALS model to the data
model = als.fit(ratings_df)

# Evaluate the model by computing the RMSE on the same data
predictions = model.transform(ratings_df)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
                                predictionCol="prediction")
rmse = evaluator.evaluate(predictions)

print("Root-mean-square error = " + str(rmse))

In [36]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [37]:
paramGrid = ParamGridBuilder() \
    .addGrid(als.maxIter, [5, 10]) \
    .addGrid(als.regParam, [0.05, 0.1]) \
    .addGrid(als.rank, [10, 15]) \
    .build()

In [38]:
# Update CrossValidator setup
crossval = CrossValidator(estimator=als,
                          estimatorParamMaps=paramGrid,
                          evaluator=RegressionEvaluator(metricName="rmse",
                                                        labelCol="rating",
                                                        predictionCol="prediction"),
                          numFolds=3)  # Use 3 folds for meaningful cross-validation

In [39]:
# Fit the model using CrossValidator
cvModel = crossval.fit(ratings_df)

# Extract the best model from the cross validation process
bestModel = cvModel.bestModel

In [ ]:
# Make predictions with the best model on the original dataset
bestPredictions = bestModel.transform(ratings_df)

# Evaluate the best model
bestRmse = evaluator.evaluate(bestPredictions)
print("Best Root-mean-square error = " + str(bestRmse))

In [41]:
# Example: Save the best model
bestModel.write().overwrite().save("als_model_v2")

In [42]:
#spark.stop()